![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [33]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None) 
cc_apps.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


In [34]:
cc_apps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       690 non-null    object 
 1   1       690 non-null    object 
 2   2       690 non-null    float64
 3   3       690 non-null    object 
 4   4       690 non-null    object 
 5   5       690 non-null    object 
 6   6       690 non-null    object 
 7   7       690 non-null    float64
 8   8       690 non-null    object 
 9   9       690 non-null    object 
 10  10      690 non-null    int64  
 11  11      690 non-null    object 
 12  12      690 non-null    int64  
 13  13      690 non-null    object 
dtypes: float64(2), int64(2), object(10)
memory usage: 75.6+ KB


In [35]:
cc_apps.isnull().sum()
X = cc_apps.iloc[:, :-1]
y = cc_apps.iloc[:, -1]
print(X.shape)
print(y.shape)
print(y.value_counts())
X.dtypes
(X == "?").sum()
X = X.replace("?", np.nan)
X.isnull().sum()

(690, 13)
(690,)
-    383
+    307
Name: 13, dtype: int64


0     12
1     12
2      0
3      6
4      6
5      9
6      9
7      0
8      0
9      0
10     0
11     0
12     0
dtype: int64

In [41]:
#Prepare and split the dataset
X = cc_apps.iloc[:, :-1].replace("?", np.nan).copy()
y = cc_apps.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Convert numeric strings to actual numbers
numerical_cols = [1, 2, 7, 10, 12]
for col in numeric_cols:
    print('currently processing column:' ,col)
    X_train[col]=pd.to_numeric(
        X_train[col], errors='coerce'
    )
    X_test[col]=pd.to_numeric(
        X_test[col], errors='coerce'
    )
    print("Finished converting column:", col)
    print("----------------------")
#Preprocessing
categorical_cols = [col for col in X.columns
                    if col not in numerical_cols]
print(categorical_cols)

#Fill missing numerical values
numerical_transformer= Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])
categorical_transformer=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

#combine numerical and categorical pipelines
preprocessor=ColumnTransformer([
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, Categorical_cols)
])

model=Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=10000))
])
model.fit(X_train,y_train)

y_predictions=model.predict(X_test)
correct_predictions=(y_test==y_predictions).sum()
best_score = correct_predictions/len(y_test)
print('Test Accuracy:', best_score)

currently processing column: 1
Finished converting column: 1
----------------------
currently processing column: 2
Finished converting column: 2
----------------------
currently processing column: 7
Finished converting column: 7
----------------------
currently processing column: 10
Finished converting column: 10
----------------------
currently processing column: 12
Finished converting column: 12
----------------------
[0, 3, 4, 5, 6, 8, 9, 11]
Test Accuracy: 0.8260869565217391
